# Notebook 56 — Fluent DSL Workflow Builder

**Module:** `multigen.dsl`

The DSL module lets you define complex agentic workflows using a **fluent builder API** — no raw dicts, no YAML, just Python method chains.

**Key builders:**
- `WorkflowBuilder` — sequential, parallel, conditional steps
- `GraphBuilder` — node/edge DAG with cycles, reflection, circuit breakers
- `branch()`, `when()`, `otherwise()` — DSL helpers

**Why it matters:** Eliminates the cognitive overhead of manually constructing workflow definition dicts. Catches misconfiguration at build time, not runtime.

In [ ]:
import importlib.util, pathlib

def load_module(name):
    path = pathlib.Path('../sdk/multigen') / f'{name}.py'
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

dsl = load_module('dsl')
print('DSL helpers:', [x for x in dir(dsl) if not x.startswith('_')])

## 1. Sequential Workflow

In [ ]:
# Build a simple sequential pipeline
wf = (
    dsl.WorkflowBuilder()
    .step('ingest',  agent='DataIngester',  params={'format': 'json'})
    .step('clean',   agent='DataCleaner',   params={'drop_nulls': True})
    .step('enrich',  agent='DataEnricher',  params={'lookup': 'geo'})
    .step('output',  agent='ReportWriter',  params={'format': 'pdf'})
    .build()
)

import json
print(json.dumps(wf, indent=2))

## 2. Parallel Fan-Out

In [ ]:
# Parallel branches run concurrently then merge
wf2 = (
    dsl.WorkflowBuilder()
    .step('fetch', agent='DataFetcher')
    .parallel('analyse',
        dsl.branch('sentiment',  agent='SentimentAgent'),
        dsl.branch('entities',   agent='NERAgent'),
        dsl.branch('summary',    agent='SummaryAgent'),
    )
    .step('merge', agent='MergeAgent')
    .build()
)

for step in wf2['steps']:
    print(f"Step: {step['name']} | type={step.get('type','sequential')} | agent={step.get('agent', '[parallel]')}")
    if 'branches' in step:
        for b in step['branches']:
            print(f"  branch: {b['name']} -> {b['agent']}")

## 3. Conditional Routing

In [ ]:
# Conditional step: route based on credit score
wf3 = (
    dsl.WorkflowBuilder()
    .step('score', agent='CreditScorer')
    .conditional('route',
        dsl.when('score.value >= 750', agent='PremiumApprove', name='premium'),
        dsl.when('score.value >= 600', agent='StandardApprove', name='standard'),
        dsl.otherwise(agent='ManualReview', name='manual'),
    )
    .step('notify', agent='NotificationAgent')
    .build()
)

route_step = next(s for s in wf3['steps'] if s['name'] == 'route')
print('Conditional step type:', route_step['type'])
for branch in route_step.get('branches', []):
    cond = branch.get('condition', 'else')
    agent = branch.get('then', branch).get('agent', branch.get('agent','?'))
    print(f'  {cond!r:40s} -> {agent}')

## 4. Graph Builder — DAG with Cycles

In [ ]:
# Build a graph with a reflection loop
graph = (
    dsl.GraphBuilder()
    .node('plan',   agent='PlannerAgent',  reflection_threshold=0.75)
    .node('act',    agent='ActorAgent')
    .node('check',  agent='CritiqueAgent', fallback_agent='FallbackCritique')
    .edge('plan',   'act')
    .edge('act',    'check')
    .edge('check',  'plan', condition='check.confidence < 0.9')
    .entry('plan')
    .max_cycles(5)
    .build()
)

print('Graph nodes:')
for n in graph['nodes']:
    print(f"  {n['id']:10s} agent={n['agent']}", end='')
    if n.get('reflection_threshold'):
        print(f" reflection={n['reflection_threshold']}", end='')
    if n.get('fallback_agent'):
        print(f" fallback={n['fallback_agent']}", end='')
    print()

print('\nEdges:')
for e in graph['edges']:
    cond = f" [if {e['condition']}]" if e.get('condition') else ''
    print(f"  {e['from']:8s} --> {e['to']}{cond}")

print(f"\nEntry: {graph['entry']}, Max cycles: {graph['max_cycles']}")

## 5. GraphBuilder — Bound Node Builder (Fluent Node Configuration)

In [ ]:
# Use .begin_node() for rich per-node configuration
graph2 = (
    dsl.GraphBuilder()
    .begin_node('research')
        .agent('ResearchAgent')
        .reflection_threshold(0.80)
        .max_retries(3)
        .done()
    .begin_node('write')
        .agent('WritingAgent')
        .max_retries(2)
        .done()
    .begin_node('review')
        .agent('ReviewAgent')
        .fallback('BasicReviewAgent')
        .done()
    .edge('research', 'write')
    .edge('write', 'review')
    .edge('review', 'research', condition='review.score < 0.8')
    .entry('research')
    .max_cycles(3)
    .circuit_breaker(trip_threshold=3, recovery_executions=5)
    .build()
)

print('Circuit breaker config:', graph2['circuit_breaker'])
print('Nodes:')
for n in graph2['nodes']:
    print(f"  {n}")

## 6. Real-World: Multi-Stage Document Processing Pipeline

In [ ]:
# A production-grade document processing pipeline
doc_pipeline = (
    dsl.WorkflowBuilder()
    # Stage 1: ingest
    .step('fetch_doc', agent='DocFetcher', params={'format': 'pdf', 'ocr': True})
    # Stage 2: parallel analysis
    .parallel('analyse',
        dsl.branch('classify',  agent='ClassifierAgent',  params={'taxonomy': 'legal'}),
        dsl.branch('extract',   agent='ExtractorAgent',   params={'fields': ['date','parties','value']}),
        dsl.branch('risk_scan', agent='RiskScanAgent',    params={'rules': 'pci_gdpr'}),
    )
    # Stage 3: conditional action
    .conditional('decide',
        dsl.when('risk_scan.risk_level == high', agent='EscalateAgent', name='escalate'),
        dsl.when('classify.category == contract', agent='ContractAgent', name='contract'),
        dsl.otherwise(agent='ArchiveAgent', name='archive'),
    )
    # Stage 4: always notify
    .step('notify', agent='NotifyAgent', params={'channel': 'slack'})
    .build()
)

print(f'Pipeline steps: {len(doc_pipeline["steps"])}')
for step in doc_pipeline['steps']:
    step_type = step.get('type', 'sequential')
    print(f"  [{step_type:12s}] {step['name']}")